# Exploratory Data Analysis (EDA) & Data Preparation - Skripsi

Notebook ini berfungsi secara khusus sebagai lampiran akademis untuk Bab 3 dan Bab 4 Skripsi. Notebook ini **tidak terhubung** dan **tidak mempengaruhi** sistem utama (`app.py`) yang dideploy di Streamlit.

Tujuan utama notebook ini adalah untuk membuktikan secara formal proses inspeksi data, pencarian nilai kosong (*missing values*), pengecekan duplikat, statistika deskriptif, dan simulasi proses Data Preparation.

## 1. Import Library
Menggunakan library standar yang sudah ada di `requirements.txt`.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler

## 2. Memuat Data
Membaca data historis saham BBCA dari file CSV cadangan yang sudah tersedia di direktori proyek.

In [ ]:
# Membaca data CSV
df = pd.read_csv('bbca_cadangan.csv')

# Mengubah tipe kolom Date menjadi datetime
df['Date'] = pd.to_datetime(df['Date'])

# Menampilkan 5 baris teratas untuk pratinjau (Preview)
df.head()

## 3. Pengecekan Informasi dan Tipe Data
Langkah ini untuk memastikan kolom sudah berada dalam tipe data yang benar.

In [ ]:
df.info()

## 4. Pengecekan Data Kosong (Missing Values)
Memeriksa apakah terdapat celah hari bursa atau nilai harga yang tidak terekam.

In [ ]:
print("Jumlah Data Kosong pada setiap kolom:")
df.isnull().sum()

## 5. Pengecekan Data Duplikat
Memastikan tidak ada baris data transaksi yang tercatat dua kali pada hari yang sama.

In [ ]:
jumlah_duplikat = df.duplicated().sum()
print(f"Jumlah baris data yang terduplikasi: {jumlah_duplikat}")

## 6. Statistika Deskriptif
Mendapatkan gambaran matematis dari pergerakan harga saham selama 10 tahun.

In [ ]:
# Hanya menampilkan statistika deskriptif untuk data numerik (Harga)
df.describe()

## 7. Visualisasi Tren Data Historis
Visualisasi pergerakan harga saham BBCA selama 10 tahun untuk melihat tren naik (uptrend) atau turun (downtrend) secara keseluruhan.

In [ ]:
fig_trend = go.Figure()
fig_trend.add_trace(go.Scatter(
    x=df['Date'], 
    y=df['Close'], 
    name="Harga Penutupan (Close)", 
    line_color='deepskyblue'
))
fig_trend.update_layout(
    title="Grafik Historis Pergerakan Harga Saham BBCA (2016-2026)", 
    xaxis_title="Tanggal",
    yaxis_title="Harga (Rupiah)"
)
fig_trend.show()

## 8. Visualisasi Pengecekan Outlier (Nilai Ekstrem)
Memvisualisasikan distribusi harga saham menggunakan *Boxplot* untuk mendeteksi adanya *outlier*.

*Catatan: Dalam prediksi harga saham, outlier seringkali merepresentasikan anomali pasar nyata sehingga sengaja dibiarkan (tidak dihapus).*

In [ ]:
fig_outlier = px.box(df, y='Close', title='Boxplot Harga Penutupan Saham BBCA (Deteksi Outlier)')
fig_outlier.update_layout(yaxis_title="Harga (Rupiah)")
fig_outlier.show()

## 9. Simulasi Data Preparation (Khusus untuk Laporan Skripsi)
Bagian ini mendemonstrasikan tahapan *Data Preparation* yang terjadi di balik layar model agar dapat di-*screenshot* untuk Bab 4.

In [ ]:
print("--- TAHAPAN DATA PREPARATION ---\n")

# A. Pemisahan Data (80% Latih, 20% Uji)
total_data = len(df)
train_size = int(total_data * 0.8)
test_size = total_data - train_size
print(f"1. Pemisahan Data (Data Splitting 80/20):")
print(f"   - Total Keseluruhan Data : {total_data} baris")
print(f"   - Total Data Latih (80%) : {train_size} baris (digunakan untuk training)")
print(f"   - Total Data Uji (20%)   : {test_size} baris (digunakan untuk evaluasi/testing)\n")

# B. Normalisasi Data untuk LSTM (MinMaxScaler)
scaler = MinMaxScaler(feature_range=(0, 1))
data_close = df[['Close']].values
scaled_data = scaler.fit_transform(data_close)
print("2. Normalisasi Data untuk LSTM (MinMaxScaler):")
print("   Tujuan: Mengubah skala ratusan ribu Rupiah menjadi angka rentang 0 hingga 1 agar LSTM lebih cepat belajar.")
print("   - 5 Baris Awal Harga Asli (Rp) :", data_close[:5].flatten())
print("   - 5 Baris Awal Harga Normalisasi :", scaled_data[:5].flatten(), "\n")

# C. Penyesuaian Kolom untuk Facebook Prophet
df_prophet = df[['Date', 'Close']].copy()
df_prophet.rename(columns={'Date': 'ds', 'Close': 'y'}, inplace=True)
print("3. Penyesuaian Format Kolom untuk Facebook Prophet:")
print("   Tujuan: Menyesuaikan nama kolom sesuai dengan standar wajib algoritma Prophet.")
print("   - Tampilan kolom setelah diubah (Date -> ds, Close -> y):")
print(df_prophet.head())